# VQMS Reviewer Copilot - MCP Demo

## What this notebook is

This notebook is a **runnable proof-of-concept** for the *Path C - Low-Confidence Human Review* workflow in the VQMS (Vendor Query Management System) Agentic AI Platform. It shows how a human reviewer, sitting in front of an ambiguous vendor query the AI couldn't confidently classify, can drive an LLM-powered "copilot" that reaches into VQMS data sources (Salesforce, ServiceNow, episodic memory, the KB) to gather the exact context needed to make a correction.

The whole demo is wired up over the **Model Context Protocol (MCP)** - the tools live on an MCP server, the reviewer agent talks to them as an MCP client, and the LLM (gpt-4o-mini here, Claude Sonnet 3.5 in production) decides which tools to call.

---

## Why MCP (and not a direct LangChain tool list)

In a normal LangChain/LangGraph setup you would just hand the agent a Python list of `@tool`-decorated functions and call it a day. We deliberately don't do that here. Instead we:

1. Define each tool as a normal `@tool` function.
2. Convert each tool into a **FastMCP** tool via `to_fastmcp(...)`.
3. Run those tools as an **MCP server** over `streamable-http`.
4. Have the reviewer agent connect to that server as an **MCP client** and load the tool schemas dynamically.

Why go through all that?

| Reason | What it gives us |
|--------|------------------|
| **Decoupling** | The reviewer copilot can run on a different process, machine, or even language than the tools. |
| **Multi-client reuse** | The same MCP server can be used by Claude Desktop, Cursor, the Angular triage portal's copilot panel, and an automated agent - all at once. |
| **Standardized contract** | Every tool exposes a JSON Schema description; any MCP-compatible client can introspect and use them without bespoke glue code. |
| **Production fit** | In real VQMS, the MCP server is the safe, audited gateway to Salesforce / ServiceNow / pgvector. Reviewers (humans + agents) only ever talk to that gateway, never to the underlying systems directly. |
| **Easy to add tools later** | Drop in another `@tool`, register it, restart the server. No client-side change required - the client re-loads the tool list at session init. |

---

## The use case - Path C triage

In VQMS, every incoming vendor query (email or portal) goes through **LLM Call #1 (Query Analysis Node)** which produces an `AnalysisResult` with a `confidence_score` between 0.0 and 1.0.

- `confidence_score >= 0.85` -> continue automatically (Path A or Path B)
- `confidence_score <  0.85` -> **Path C: workflow PAUSES** and a human reviewer must investigate before anything else happens

When a reviewer picks up a Path C ticket, they need to answer three questions fast:

1. **Why was the AI uncertain?** (which dimensions scored low - intent? entities? vendor context?)
2. **Who is this vendor and what's their history with us?** (tier, recent queries, open tickets)
3. **What should this be classified as, and which team should own it?**

Without tooling, the reviewer would have to open Salesforce in one tab, ServiceNow in another, run a SQL query against `memory.episodic_memory`, scroll through KB articles, and stitch the picture together themselves. That's slow and error-prone, especially under SLA pressure.

The **Reviewer Copilot** in this notebook does that stitching for them. The reviewer asks a plain-English question; the LLM decides which MCP tools to call, calls them in order, and produces a single concise recommendation with vendor context, ticket history, similar past queries, and KB references.

---

## Architecture at a glance

```
+----------------------------------------------------------------+
|  Reviewer (human, in the Angular triage portal)                |
|      |                                                         |
|      |  "Help me figure out what VQ-2026-0123 is about"        |
|      v                                                         |
|  +----------------------------------------------------------+  |
|  |  Reviewer Copilot Agent  (LangGraph create_react_agent)  |  |
|  |  - LLM: gpt-4o-mini  (Claude Sonnet 3.5 in prod)         |  |
|  |  - System prompt: "you are the VQMS triage copilot..."   |  |
|  +----------------------------------------------------------+  |
|      |  MCP client (streamable-http)                           |
|      v                                                         |
|  +----------------------------------------------------------+  |
|  |  FastMCP server  (http://127.0.0.1:8765/mcp)             |  |
|  |  Tools exposed over MCP:                                 |  |
|  |   1. confidence_breakdown_explainer  -> workflow.case_ex |  |
|  |   2. vendor_lookup                   -> Salesforce       |  |
|  |   3. episodic_memory_for_vendor      -> memory.episodic  |  |
|  |   4. get_similar_past_queries        -> pgvector cos.sim |  |
|  |   5. view_servicenow_history         -> ServiceNow       |  |
|  |   6. kb_search                       -> memory.embedding |  |
|  +----------------------------------------------------------+  |
|      |   In this notebook each tool reads in-memory dicts.     |
|      |   In production each tool wraps the real adapter        |
|      |   (src/adapters/salesforce/, src/adapters/servicenow/,  |
|      |    src/db/connection/).                                 |
+----------------------------------------------------------------+
```

---

## What each cell does (top to bottom)

1. **Imports** - LangChain LLM bindings, MCP server (`FastMCP`) and client (`ClientSession`, `streamablehttp_client`), the LangChain<->MCP bridge (`to_fastmcp`, `load_mcp_tools`), and `create_react_agent` for the reviewer agent loop.
2. **Mock data** - In-memory stand-ins for Salesforce, ServiceNow, `memory.episodic_memory`, `memory.embedding_index`, and one `workflow.case_execution` row containing a real-looking Path C `AnalysisResult` (confidence 0.62, ambiguous intent, no entities).
3. **LLM config** - Cheap, deterministic model behind a Great Learning gateway. Swap to `BedrockChat(model_id="anthropic.claude-3-5-sonnet-...")` for production.
4. **Tools** - Six `@tool` functions, one per data source. Each docstring is what the LLM actually reads to decide whether to call it - they're written to be self-explanatory and to nudge the agent toward the right call order.
5. **Convert tools -> FastMCP and start the server** - Wrap each LangChain tool with `to_fastmcp`, hand them to `FastMCP("VQMS Reviewer Copilot", ...)`, and run the server in a background daemon thread on port 8765 over `streamable-http` transport.
6. **Reviewer client (the agent loop)** - `run_reviewer_copilot` opens an MCP `streamablehttp_client`, initializes a `ClientSession`, calls `load_mcp_tools(session)` to introspect the server, and gives those tools to a `create_react_agent`. The agent then ReAct-loops (think -> call tool -> observe -> think) until it has enough context, then writes a final recommendation. The system prompt enforces guard-rails: max 4 tool calls, no duplicate calls, must call `confidence_breakdown_explainer` first, final answer < 200 words.
7. **Three demo runs** - Same query (`VQ-2026-0123`) but three different reviewer questions, showing how the agent picks different tool combinations depending on what the reviewer asks.

---

## How this maps to real VQMS

| Demo piece | Production equivalent |
|---|---|
| `VENDOR_DB` dict | `src/adapters/salesforce/vendor_lookup.py` (with `cache.vendor_cache` 1h TTL) |
| `EPISODIC_MEMORY_DB` list | `memory.episodic_memory` table queried via `src/db/connection/` |
| `SERVICENOW_TICKETS` list | `src/adapters/servicenow/ticket_query.py` |
| `KB_ARTICLES` + TF-IDF | `memory.embedding_index` (pgvector) + Bedrock Titan Embed v2, 1536-dim cosine search inside `kb_search` node |
| `QUERY_ANALYSIS_DB` row | `workflow.case_execution.analysis_result` JSONB column written by `src/orchestration/nodes/query_analysis.py` |
| FastMCP background thread | A separately deployed FastMCP `uvicorn` process behind the same VPC, talking to real Salesforce / ServiceNow / RDS |
| Notebook reviewer agent | Triage portal's "Copilot" panel (Phase 7 Reviewer Module) calling the same MCP server |

The point of the notebook is to show that the agent loop, the tool docstrings, and the system prompt are all that change between this demo and the production reviewer copilot - the MCP boundary stays identical.


### Imports

In [1]:
import os
import json
import asyncio
import threading
import time
from datetime import datetime

# LLM and message types
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# MCP server side
from mcp.server.fastmcp import FastMCP
from langchain_core.tools import tool
from langchain_mcp_adapters.tools import to_fastmcp

# MCP client side
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.tools import load_mcp_tools

# ReAct agent (the "reviewer copilot brain")
from langgraph.prebuilt import create_react_agent

# For the mock semantic search inside kb_search and get_similar_past_queries.
# In real VQMS, these calls go to pgvector via memory.embedding_index.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Mock data (this stands in for your real datastores)

In [2]:
# ─────────────────────────────────────────────────────────
# Stands in for Salesforce vendor master.
# In real VQMS: src/adapters/salesforce/vendor_lookup.py → SOQL on Account.
# ─────────────────────────────────────────────────────────
VENDOR_DB = {
    "VEND-001": {
        "vendor_id": "VEND-001",
        "company_name": "TechNova Solutions",
        "tier": "Gold",                       # Drives SLA — Gold = stricter
        "primary_contact": "rajesh.kumar@technova.com",
        "active_since": "2022-03-15",
        "status": "active",
        "industry": "IT Services",
        "annual_spend_usd": 1_250_000,
        "account_manager": "Priya Sharma",
    },
    "VEND-002": {
        "vendor_id": "VEND-002",
        "company_name": "Delta Foods",
        "tier": "Silver",
        "primary_contact": "supplychain@deltafoods.com",
        "active_since": "2023-08-01",
        "status": "active",
        "industry": "Food & Beverage",
        "annual_spend_usd": 480_000,
        "account_manager": "Anil Verma",
    },
}

# ─────────────────────────────────────────────────────────
# Stands in for memory.episodic_memory table.
# Each row = one past vendor interaction summary.
# ─────────────────────────────────────────────────────────
EPISODIC_MEMORY_DB = [
    {"vendor_id": "VEND-001", "query_id": "VQ-2026-0089", "date": "2026-04-10",
     "intent": "invoice_status", "path": "A", "outcome": "resolved", "satisfaction": 5,
     "summary": "Invoice INV-7741 status query, AI resolved with payment date."},
    {"vendor_id": "VEND-001", "query_id": "VQ-2026-0102", "date": "2026-04-15",
     "intent": "po_amendment", "path": "B", "outcome": "resolved", "satisfaction": 4,
     "summary": "PO-2299 quantity change request, procurement team investigated and approved."},
    {"vendor_id": "VEND-001", "query_id": "VQ-2026-0118", "date": "2026-04-20",
     "intent": "payment_terms_query", "path": "A", "outcome": "resolved", "satisfaction": 5,
     "summary": "Asked about NET-30 vs NET-45 terms, AI cited contract Section 4.2."},
    {"vendor_id": "VEND-002", "query_id": "VQ-2026-0091", "date": "2026-04-11",
     "intent": "delivery_delay", "path": "B", "outcome": "resolved", "satisfaction": 3,
     "summary": "Shipment delayed two weeks, team coordinated with logistics."},
]

# ─────────────────────────────────────────────────────────
# Stands in for ServiceNow incident table.
# In real VQMS: src/adapters/servicenow/ticket_query.py → REST GET /api/now/table/incident
# ─────────────────────────────────────────────────────────
SERVICENOW_TICKETS = [
    {"ticket_id": "INC-7723451", "vendor_id": "VEND-001", "query_id": "VQ-2026-0089",
     "status": "RESOLVED", "priority": "MEDIUM", "team": "AP-FINANCE",
     "opened": "2026-04-10", "closed": "2026-04-10",
     "resolution_summary": "Provided payment date and reference number."},
    {"ticket_id": "INC-7724102", "vendor_id": "VEND-001", "query_id": "VQ-2026-0102",
     "status": "RESOLVED", "priority": "HIGH", "team": "PROCUREMENT",
     "opened": "2026-04-15", "closed": "2026-04-17",
     "resolution_summary": "PO amended; updated quantities sent for re-approval."},
    {"ticket_id": "INC-7725012", "vendor_id": "VEND-001", "query_id": "VQ-2026-0123",
     "status": "OPEN", "priority": "MEDIUM", "team": "UNASSIGNED",
     "opened": "2026-04-25", "closed": None, "resolution_summary": None},
]

# ─────────────────────────────────────────────────────────
# Stands in for memory.embedding_index (KB articles + their pgvector embeddings).
# In real VQMS: src/orchestration/nodes/kb_search.py
# embeds the query via Bedrock Titan v2 then runs cosine similarity in PostgreSQL.
# We use TF-IDF here so the demo works without Bedrock.
# ─────────────────────────────────────────────────────────
KB_ARTICLES = [
    {"article_id": "KB-101", "category": "invoicing",
     "title": "How to Check Invoice Payment Status",
     "content": "Vendors can check invoice payment status by referencing PO number. AP processes payments per agreed NET terms. Standard NET-30 = payment within 30 days of invoice receipt."},
    {"article_id": "KB-102", "category": "invoicing",
     "title": "Disputing an Invoice Amount",
     "content": "If you believe an invoice was incorrectly paid or rejected, submit a dispute via the vendor portal with backup. Disputes reviewed in 5 business days by AP."},
    {"article_id": "KB-201", "category": "procurement",
     "title": "PO Amendment Process",
     "content": "PO amendments need formal approval. Submit changes via the portal with justification. Quantity changes require re-approval if value increases by more than 10%."},
    {"article_id": "KB-301", "category": "logistics",
     "title": "Delivery Delay Reporting",
     "content": "Report delivery delays via the vendor portal as soon as a delay is anticipated. Penalties may apply per contract Section 7.3."},
    {"article_id": "KB-501", "category": "payments",
     "title": "Banking Detail Update Process",
     "content": "Bank account changes require dual verification: written request plus phone confirmation by registered finance contact. Processed in 3 business days."},
]

# ─────────────────────────────────────────────────────────
# Stands in for the workflow.case_execution row + the AnalysisResult JSON
# produced by Step 8 (Query Analysis Node, LLM Call #1).
# This is THE row a Path C reviewer will be investigating.
# ─────────────────────────────────────────────────────────
QUERY_ANALYSIS_DB = {
    "VQ-2026-0123": {
        "query_id": "VQ-2026-0123",
        "vendor_id": "VEND-001",
        "received_at": "2026-04-25T09:14:00+05:30",
        "subject": "Question about our payment situation",
        "body": ("Hi, regarding our recent dealings, can you help us figure out where things stand? "
                 "It's getting confusing on our end. Need clarity ASAP."),
        "ai_analysis": {
            "intent_classification": "ambiguous",
            "extracted_entities": {},                  # Empty = no invoice/PO found
            "urgency_level": "MEDIUM",
            "sentiment": "frustrated",
            "confidence_score": 0.62,                  # Below 0.85 threshold → Path C
            "multi_issue_detected": True,
            "suggested_category": "payments",
            # Per-dimension breakdown (this is what a reviewer wants to see)
            "confidence_breakdown": {
                "intent_clarity": 0.45,                 # Vague language
                "entity_extraction": 0.40,              # No structured entities
                "category_match": 0.78,
                "vendor_context_alignment": 0.85,
            },
            "low_confidence_reasons": [
                "Query uses vague phrases like 'recent dealings' and 'where things stand' "
                "without specific entities (no invoice number, PO number, or amount).",
                "'It's getting confusing on our end' suggests multiple unrelated concerns "
                "may be bundled together.",
                "Sentiment is 'frustrated' which often indicates the vendor expects context "
                "the AI doesn't have.",
            ],
        },
    }
}


### LLM config

In [3]:
openai_api_key = "gl-U2FsdGVkX18MRD/3eUPfiSOXGn2UTdfXXQnQgBnYRYM9FPGPh3Hc92WSu4X14Koz"

llm = ChatOpenAI(
    api_key=openai_api_key,
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    model='gpt-4o-mini',
    temperature=0,                # Deterministic for reviewer use case
)


In [4]:
# # from langchain_aws import BedrockChat

# # llm = BedrockChat(
# #     model_id="anthropic.claude-3-sonnet-20241022-v2:0",  # Claude 3 Sonnet
# #     region_name="us-east-1",                             # Your AWS region
# #     temperature=0,                                       # Deterministic for reviewer use case
# )

### Tools

In [1]:
# Tool #1: Given a query ID, fetch the AI's confidence breakdown so the reviewer knows
# why it got flagged for human review.


@tool
def confidence_breakdown_explainer(query_id: str) -> str:
    """Explain why the AI assigned low confidence to a specific Path C query.

    Use this FIRST when a reviewer opens a triage case — it tells you what the
    AI struggled with so you know which other tools to call next.

    Args:
        query_id: The VQMS query ID, e.g. 'VQ-2026-0123'.

    Returns:
        JSON string with overall confidence, the 4-dimension breakdown,
        human-readable reasons the AI was uncertain, and the original query subject.
    """
    # Production: SELECT analysis_result FROM workflow.case_execution WHERE query_id = $1
    analysis = QUERY_ANALYSIS_DB.get(query_id)
    if not analysis:
        return json.dumps({"error": f"Query {query_id} not found"})

    ai = analysis["ai_analysis"]
    return json.dumps({
        "query_id": query_id,
        "vendor_id": analysis["vendor_id"],
        "subject": analysis["subject"],
        "body_excerpt": analysis["body"][:300],
        "overall_confidence": ai["confidence_score"],
        "threshold": 0.85,
        "decision": "Path C (human review)" if ai["confidence_score"] < 0.85 else "Path A or B",
        "intent": ai["intent_classification"],
        "extracted_entities": ai["extracted_entities"],
        "confidence_breakdown": ai["confidence_breakdown"],
        "low_confidence_reasons": ai["low_confidence_reasons"],
    })


NameError: name 'tool' is not defined

In [ ]:
# Given a vendor ID, fetch their Salesforce profile so
# the reviewer knows who they are and how important they are.

@tool
def vendor_lookup(vendor_id: str) -> str:
    """Get the full vendor profile from Salesforce.

    Use this to understand vendor context: tier (drives SLA), industry,
    annual spend (drives priority), account manager (escalation path).

    Args:
        vendor_id: Salesforce vendor ID, e.g. 'VEND-001'.

    Returns:
        JSON with company_name, tier, primary_contact, active_since, status,
        industry, annual_spend_usd, account_manager.
    """
    # Production: src/adapters/salesforce/vendor_lookup.py → simple-salesforce SOQL.
    # Uses cache.vendor_cache (1h TTL) before hitting Salesforce.
    vendor = VENDOR_DB.get(vendor_id)
    if not vendor:
        return json.dumps({"error": f"Vendor {vendor_id} not found"})
    return json.dumps(vendor)


In [ ]:
# Given a vendor ID, fetch their recent query history so the reviewer can see patterns and past outcomes.

@tool
def episodic_memory_for_vendor(vendor_id: str, limit: int = 5) -> str:
    """Get this vendor's recent past interactions.

    Use this when you need to understand patterns: do they always ask about
    invoices? Have past queries gone Path B (team-handled)? What was their
    last satisfaction score?

    Args:
        vendor_id: The vendor ID.
        limit: Max past interactions (default 5).

    Returns:
        JSON list ordered newest-first with query_id, date, intent, path,
        outcome, satisfaction, summary.
    """
    # Production: SELECT * FROM memory.episodic_memory
    #             WHERE vendor_id = $1 ORDER BY date DESC LIMIT $2
    history = [m for m in EPISODIC_MEMORY_DB if m["vendor_id"] == vendor_id]
    history = sorted(history, key=lambda x: x["date"], reverse=True)[:limit]
    return json.dumps({"vendor_id": vendor_id, "history": history, "count": len(history)})


In [ ]:
# Given a query's text, fetch the most 
# similar past resolved queries so the reviewer can copy what worked before.

@tool
def get_similar_past_queries(query_text: str, top_k: int = 3) -> str:
    """Find resolved queries similar to the given text, regardless of vendor.

    Use this when you have an ambiguous query and want to see how similar
    queries were classified and resolved historically.

    Args:
        query_text: The new query's text (subject + body works best).
        top_k: How many similar past queries to return.

    Returns:
        JSON ranked list with query_id, intent, path taken, summary, similarity score.
    """
    # Production: embed query_text via Bedrock Titan v2 (1536-dim),
    # then cosine search against memory.embedding_index filtered by outcome='resolved'.
    # Here we use TF-IDF as a stand-in.
    resolved = [m for m in EPISODIC_MEMORY_DB if m["outcome"] == "resolved"]
    if not resolved:
        return json.dumps({"results": []})

    summaries = [m["summary"] for m in resolved]
    vectorizer = TfidfVectorizer().fit(summaries + [query_text])
    summary_vectors = vectorizer.transform(summaries)
    query_vector = vectorizer.transform([query_text])
    scores = cosine_similarity(query_vector, summary_vectors)[0]

    ranked = sorted(zip(resolved, scores), key=lambda x: x[1], reverse=True)[:top_k]
    results = [{
        "query_id": m["query_id"],
        "intent": m["intent"],
        "path": m["path"],
        "summary": m["summary"],
        "similarity": round(float(s), 3),
    } for m, s in ranked]
    return json.dumps({"results": results})


In [ ]:
# Given a vendor ID, fetch their full ServiceNow ticket history so the
# reviewer can check for duplicates and past handling.

@tool
def view_servicenow_history(vendor_id: str) -> str:
    """List past ServiceNow tickets for a vendor.

    Use this to see if there are open tickets (potential thread/duplicate),
    which teams have handled this vendor before, and resolution patterns.

    Args:
        vendor_id: The vendor ID.

    Returns:
        JSON list of tickets newest-first with ticket_id, status, priority,
        team, opened/closed dates, resolution summary.
    """
    # Production: src/adapters/servicenow/ticket_query.py
    # GET /api/now/table/incident?sysparm_query=u_vendor_id={vendor_id}
    tickets = [t for t in SERVICENOW_TICKETS if t["vendor_id"] == vendor_id]
    tickets = sorted(tickets, key=lambda x: x["opened"], reverse=True)
    return json.dumps({"vendor_id": vendor_id, "tickets": tickets, "count": len(tickets)})


In [ ]:
# Given a free-text query (and optional category), search the knowledge base
# so the reviewer can find the right article the AI missed.

@tool
def kb_search(query: str, category: str = "", top_k: int = 3) -> str:
    """Search the knowledge base by semantic similarity.

    Use this when you suspect the AI's KB search missed something — the
    reviewer can refine the query text and re-run the search with a category filter.

    Args:
        query: Free-text query.
        category: Optional filter — 'invoicing' | 'procurement' | 'logistics'
                  | 'payments' | 'compliance'. Empty string = all categories.
        top_k: Number of top results.

    Returns:
        JSON ranked list of KB articles with article_id, title, content snippet, similarity.
    """
    # Production: src/orchestration/nodes/kb_search.py logic
    # 1. Embed `query` via Bedrock Titan Embed v2 (1536 dims)
    # 2. SELECT ... FROM memory.embedding_index ORDER BY embedding <=> $1 LIMIT $2
    # 3. Optionally filter by category column.
    candidates = KB_ARTICLES
    if category:
        candidates = [a for a in candidates if a["category"] == category]
    if not candidates:
        return json.dumps({"results": []})

    docs = [f"{a['title']}. {a['content']}" for a in candidates]
    vectorizer = TfidfVectorizer().fit(docs + [query])
    doc_vectors = vectorizer.transform(docs)
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, doc_vectors)[0]

    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_k]
    results = [{
        "article_id": a["article_id"],
        "category": a["category"],
        "title": a["title"],
        "content_snippet": a["content"][:200] + "...",
        "similarity": round(float(s), 3),
    } for a, s in ranked]
    return json.dumps({"results": results})


### Convert tools to FastMCP and start the server

### How the MCP server is wired up (read this before running the next cell)

This is the core of "implementing MCP" in the notebook. Three things happen:

1. **`to_fastmcp(tool)`** - every LangChain `@tool` function we defined above is converted into a `FastMCPTool`. This bridge inspects the function's signature and docstring and turns them into the JSON Schema that MCP clients will see when they list available tools. The docstrings we wrote are not just for humans - they become the tool's `description` over the wire and are what the agent's LLM reads to decide *which tool to call and with what arguments*.

2. **`FastMCP("VQMS Reviewer Copilot", host=..., port=..., tools=[...])`** - creates the MCP server. `FastMCP` is the high-level MCP server class from the official `mcp` Python SDK. It speaks the MCP protocol (JSON-RPC over a chosen transport) and handles tool discovery (`tools/list`) and invocation (`tools/call`) for us.

3. **`mcp.run(transport="streamable-http")`** - starts the server using the **Streamable HTTP** transport, which is the modern MCP transport designed for web deployments (vs the older `stdio` transport which is for local desktop integrations like Claude Desktop). The endpoint is `http://127.0.0.1:8765/mcp`. We launch it in a daemon thread so the notebook stays interactive - in production this would be its own `uvicorn` process.

Once the server is up, **any** MCP-compatible client (Claude Desktop, Cursor, our own LangGraph agent below, a future Angular UI panel, an evaluation harness, etc.) can connect to that URL, list the six tools, and call them. That's the whole point of using MCP instead of just a Python list of tools - the tools become a network-addressable, language-agnostic capability surface.


In [ ]:
# Packages the 6 tools into an MCP server and starts it on port 8765 so the agent can call them.


# Bundle all six tools.
reviewer_tools = [
    confidence_breakdown_explainer,
    vendor_lookup,
    episodic_memory_for_vendor,
    get_similar_past_queries,
    view_servicenow_history,
    kb_search,
]

# Convert each LangChain tool into a FastMCP tool.
# This is the bridge between LangGraph-style tools and the MCP protocol.
fastmcp_tools = [to_fastmcp(t) for t in reviewer_tools]

# Create the MCP server.
mcp = FastMCP(
    "VQMS Reviewer Copilot",
    host="127.0.0.1",
    port=8765,
    tools=fastmcp_tools,
)

# Run the server in a background daemon thread so the notebook stays interactive.
# Production note: you would NOT run it in a thread — you'd run it as its own
# uvicorn process behind the same load balancer or as a sidecar.
thread = threading.Thread(
    target=mcp.run,
    kwargs={"transport": "streamable-http"},
    daemon=True,
)
thread.start()
time.sleep(2)
print("VQMS Reviewer Copilot MCP server running at http://127.0.0.1:8765/mcp")

INFO:     Started server process [27232]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)


VQMS Reviewer Copilot MCP server running at http://127.0.0.1:8765/mcp


### Reviewer client (the agent loop)

### How the MCP client + reviewer agent works

The function below is the **client side of MCP**. Here's what happens, step by step, when a reviewer asks a question:

1. **`streamablehttp_client("http://127.0.0.1:8765/mcp")`** - opens the Streamable HTTP transport to the MCP server. Returns a pair of streams (`read`, `write`) that speak the MCP wire format.

2. **`ClientSession(read, write)`** - wraps those streams in an MCP client session. The session handles the protocol-level handshake, request/response correlation, and capability negotiation.

3. **`session.initialize()`** - performs the MCP `initialize` handshake (protocol version negotiation, capability exchange). This is the moment the client and server agree they understand each other.

4. **`load_mcp_tools(session)`** - calls `tools/list` on the server, gets back the six tool schemas (name, description, JSON Schema for arguments), and wraps each one as a LangChain `BaseTool` so it can be plugged straight into a LangGraph agent. **This is the magic bridge**: an agent that was written for LangChain tools can now use any MCP server's tools without code changes.

5. **`create_react_agent(llm, tools=mcp_tools, prompt=system_prompt)`** - builds a ReAct (Reason + Act) agent. The agent will repeatedly: read the conversation, decide whether to call a tool, call it (which goes back over MCP to the server), get the result, and either call another tool or write a final answer.

6. **`agent.ainvoke({"messages": [user_msg]}, config={"recursion_limit": 12})`** - runs the loop. The recursion limit plus the system prompt's "max 4 tool calls" rule are the safety belts that keep the agent from spiraling.

The system prompt is doing a lot of work here. It enforces:

- **`confidence_breakdown_explainer` MUST be called first** - so the agent always grounds itself in *why* the AI was unsure before pulling other context.
- **Max 4 tool calls** - keeps reviewer wait time low and prevents tool-call thrashing on ambiguous questions.
- **No duplicate calls with same args** - common LLM failure mode; cheap to prevent.
- **Final answer < 200 words** - reviewers need scannable output, not essays.

The three demo runs after this cell intentionally vary the reviewer's question to show the agent picking *different* tool subsets depending on intent: a "what is this about" question pulls vendor + history + tickets; a "history" question skips the KB; a "do we have a KB article" question goes straight to `kb_search`.


In [ ]:
# _as_text — "Normalize LangChain's mixed string/list message content into a plain string for printing."
# run_reviewer_copilot — "Connect to the MCP server, run the reviewer's question
# through a rule-bound ReAct agent, and print every tool call and the final answer."


def _as_text(content) -> str:
    """msg.content can be str or list[dict] in newer LangChain — normalize."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text", "") or str(block))
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)


async def run_reviewer_copilot(query_id: str, reviewer_question: str):
    async with streamablehttp_client("http://127.0.0.1:8765/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = await load_mcp_tools(session)

            # Sanity check — guards against stale server on a shared port.
            tool_names = [t.name for t in mcp_tools]
            print(f"[Loaded MCP tools] {tool_names}\n")
            assert "confidence_breakdown_explainer" in tool_names, (
                f"Wrong MCP server. Got tools: {tool_names}. "
                "Restart kernel and re-run Cell 11."
            )

            system_prompt = (
                "You are the VQMS Triage Reviewer Copilot.\n\n"
                "RULES (follow strictly):\n"
                "1. Call AT MOST 4 tools total. After 4 tool calls you MUST stop and answer.\n"
                "2. NEVER call the same tool with the same arguments twice.\n"
                "3. Always call confidence_breakdown_explainer FIRST.\n"
                "4. Once you have enough context, STOP calling tools and write a concise "
                "   final recommendation: suggested intent, suggested team, what to ask vendor.\n"
                "5. Keep the final answer under 200 words."
            )

            agent = create_react_agent(llm, tools=mcp_tools, prompt=system_prompt)
            user_msg = HumanMessage(
                content=f"Query ID: {query_id}\n\nReviewer asks: {reviewer_question}"
            )

            try:
                result = await agent.ainvoke(
                    {"messages": [user_msg]},
                    config={"recursion_limit": 12},
                )
            except Exception as e:
                print(f"\n[ERROR] {type(e).__name__}: {e}")
                import traceback; traceback.print_exc()
                return

            print("=" * 70)
            for msg in result["messages"]:
                text = _as_text(msg.content)
                if msg.type == "human":
                    print(f"\n[REVIEWER]\n{text}\n")
                elif msg.type == "ai":
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        for tc in msg.tool_calls:
                            print(f"[AGENT calls → {tc['name']}({tc['args']})]")
                    if text:
                        print(f"\n[COPILOT]\n{text}\n")
                elif msg.type == "tool":
                    snippet = text[:400] + ("..." if len(text) > 400 else "")
                    print(f"[TOOL → {msg.name}]\n{snippet}\n")
            print("=" * 70)


In [13]:
await run_reviewer_copilot(
    query_id="VQ-2026-0123",
    reviewer_question=(
        "The AI flagged this as low confidence. Help me figure out what this vendor "
        "is actually asking about and which team should handle it."
    ),
)


Created new transport with session ID: 11abe5b2b6944c19a9cef94f5d420154


INFO:     127.0.0.1:62901 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Received session ID: 11abe5b2b6944c19a9cef94f5d420154
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:62902 - "POST /mcp HTTP/1.1" 202 Accepted


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 202 Accepted"


...

HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Processing request of type CallToolRequest
HTTP Request: POST https://aibe.mygreatlearning.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


INFO:     127.0.0.1:51285 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Processing request of type CallToolRequest
HTTP Request: POST https://aibe.mygreatlearning.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Terminating session: 11abe5b2b6944c19a9cef94f5d420154



[REVIEWER]
Query ID: VQ-2026-0123

Reviewer asks: The AI flagged this as low confidence. Help me figure out what this vendor is actually asking about and which team should handle it.

[AGENT calls → confidence_breakdown_explainer({'query_id': 'VQ-2026-0123'})]
[TOOL → confidence_breakdown_explainer]
{"query_id": "VQ-2026-0123", "vendor_id": "VEND-001", "subject": "Question about our payment situation", "body_excerpt": "Hi, regarding our recent dealings, can you help us figure out where things stand? It's getting confusing on our end. Need clarity ASAP.", "overall_confidence": 0.62, "threshold": 0.85, "decision": "Path C (human review)", "intent": "ambiguous", "extracted_entities": {}, "confid...

[AGENT calls → vendor_lookup({'vendor_id': 'VEND-001'})]
[TOOL → vendor_lookup]
{"vendor_id": "VEND-001", "company_name": "TechNova Solutions", "tier": "Gold", "primary_contact": "rajesh.kumar@technova.com", "active_since": "2022-03-15", "status": "active", "industry": "IT Services", "annual_

HTTP Request: DELETE http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"


In [14]:
await run_reviewer_copilot(
    query_id="VQ-2026-0123",
    reviewer_question="What's this vendor's history with us? Anything I should be aware of?",
)


Created new transport with session ID: 1506d13e790a46798d380ae37ce81e12


INFO:     127.0.0.1:61752 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Received session ID: 1506d13e790a46798d380ae37ce81e12
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:61753 - "POST /mcp HTTP/1.1" 202 Accepted


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 202 Accepted"


...

HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"


INFO:     127.0.0.1:61764 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Processing request of type CallToolRequest
HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Processing request of type CallToolRequest
Processing request of type CallToolRequest
HTTP Request: POST https://aibe.mygreatlearning.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Terminating session: 1506d13e790a46798d380ae37ce81e12



[REVIEWER]
Query ID: VQ-2026-0123

Reviewer asks: What's this vendor's history with us? Anything I should be aware of?

[AGENT calls → confidence_breakdown_explainer({'query_id': 'VQ-2026-0123'})]
[TOOL → confidence_breakdown_explainer]
{"query_id": "VQ-2026-0123", "vendor_id": "VEND-001", "subject": "Question about our payment situation", "body_excerpt": "Hi, regarding our recent dealings, can you help us figure out where things stand? It's getting confusing on our end. Need clarity ASAP.", "overall_confidence": 0.62, "threshold": 0.85, "decision": "Path C (human review)", "intent": "ambiguous", "extracted_entities": {}, "confid...

[AGENT calls → vendor_lookup({'vendor_id': 'VEND-001'})]
[AGENT calls → episodic_memory_for_vendor({'vendor_id': 'VEND-001', 'limit': 5})]
[AGENT calls → view_servicenow_history({'vendor_id': 'VEND-001'})]
[TOOL → vendor_lookup]
{"vendor_id": "VEND-001", "company_name": "TechNova Solutions", "tier": "Gold", "primary_contact": "rajesh.kumar@technova.com", 

HTTP Request: DELETE http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
GET stream disconnected, reconnecting in 1000ms...


In [ ]:
await run_reviewer_copilot(
    query_id="VQ-2026-0123",
    reviewer_question=(
        "If this turns out to be about banking detail changes, do we have a KB article "
        "I should reference in my correction notes?"
    ),
)


Created new transport with session ID: 03be0ba751824ced85064ae882e9b794


INFO:     127.0.0.1:56584 - "POST /mcp HTTP/1.1" 200 OK


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
Received session ID: 03be0ba751824ced85064ae882e9b794
Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:56585 - "POST /mcp HTTP/1.1" 202 Accepted


HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 202 Accepted"


...

INFO:     127.0.0.1:56594 - "POST /mcp HTTP/1.1" 200 OK


Processing request of type CallToolRequest
HTTP Request: POST http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"
HTTP Request: POST https://aibe.mygreatlearning.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Terminating session: 03be0ba751824ced85064ae882e9b794



[REVIEWER]
Query ID: VQ-2026-0123

Reviewer asks: If this turns out to be about banking detail changes, do we have a KB article I should reference in my correction notes?

[AGENT calls → confidence_breakdown_explainer({'query_id': 'VQ-2026-0123'})]
[TOOL → confidence_breakdown_explainer]
{"query_id": "VQ-2026-0123", "vendor_id": "VEND-001", "subject": "Question about our payment situation", "body_excerpt": "Hi, regarding our recent dealings, can you help us figure out where things stand? It's getting confusing on our end. Need clarity ASAP.", "overall_confidence": 0.62, "threshold": 0.85, "decision": "Path C (human review)", "intent": "ambiguous", "extracted_entities": {}, "confid...

[AGENT calls → kb_search({'query': 'banking detail changes', 'top_k': 3})]
[TOOL → kb_search]
{"results": [{"article_id": "KB-501", "category": "payments", "title": "Banking Detail Update Process", "content_snippet": "Bank account changes require dual verification: written request plus phone confirmation

HTTP Request: DELETE http://127.0.0.1:8765/mcp "HTTP/1.1 200 OK"


: 